# Example Walkthrough of MSE Model

*Please reference the following publication and associated supporting information for equations and experimental design details.*

> Thanh-Tin Nguyen, Sunil Shah, Jesse McVay, Meltem Urgun-Demirtas, and Lauren Valentino. (2026). Mechanistic mass transfer in hollow fiber membrane solvent extraction for bio-based isobutanol. *Journal of Membrane Science.* Volume **752**;125544. <https://doi.org/10.1016/j.memsci.2026.125544>.

## Setting up the data for modeling

**Step 1.** Check that file paths are correct and that functions and data import properly.

In [1]:
from pathlib import Path
import sys

src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path))

**Step 2.** Load necessary Python libraries.

In [2]:
import numpy as np
import pandas as pd
import xlwings as xw

**Step 3.** Import experimental data, constants, and python modules.

In [3]:
import evaluation, fitting, model, conversions

from model import (calculate_kvals, predict_kov)
from fitting import (get_dataset_split, compile_params, least_squares, fit_model, modify_case)
from evaluation import evaluate_model
from conversions import conversion_dict

wb = xw.Book("constants.xlsx")
exp_data = pd.read_csv("exp_data.csv")

In [4]:
exp_data

,exp_id,run,phi,a_cm2,Q_a_mLmin,Q_o_mLmin,NF,m,Kov_exp
0,1,1,0.10,11.78,12.4,5.0,25,3.92,2.540280e-07
1,1,2,0.15,17.66,16.4,7.6,38,3.92,2.486970e-07
2,1,3,0.20,23.55,20.0,10.0,50,3.92,2.373900e-07
3,2,3,0.20,23.55,20.0,10.0,50,3.92,2.373900e-07
4,2,4,0.20,23.55,30.0,10.0,50,3.92,2.749880e-07
5,2,5,0.20,23.55,40.0,10.0,50,3.92,3.032680e-07
6,2,6,0.20,23.55,50.0,10.0,50,3.92,3.236950e-07


See the table below for a description of the columns.

| Name | Description | Units |
| :----- | :----- | :----- |
| `exp_id` | experimental identifier | unitless |
| `run` | specific experiment number | unitless |
| `phi` | packing fraction, between 0 and 1 | unitless |
| `a_cm2` | effective membrane area | cm² |
| `Q_a_mLmin` | aqueous flowrate | mL/min |
| `Q_o_mLmin` | organic flowrate | mL/min |
| `NF` | number of fibers | unitless |
| `Kov_exp` | experimentally determined mass transfer coefficient | m/s |

**Step 4.** Perform unit conversions.

These are not values that are not changed across experiments. Examples include the density of solvents used, fixed aspects of membrane geometry, etc. These can be ajusted prior to running the model by accessing `constants.xlsx` and adjusting the associated cell references.

> **_Note:_** If you update the units of the input parameters, be sure to change the associated conversion in `conversions.py`.

In [5]:
cst = wb.sheets[0]

for cell, func in conversion_dict.items():
    raw = cst.range(cell).value
    cst.range(cell).offset(0, 2).value = func(raw)  

values = cst.range("E2:E14").value
(mu_a, mu_o, rho_a, rho_o, D_a, D_o, dsi, di, do, L, epsilon, tau, l) = values

constants = {
    "mu_a": mu_a,
    "mu_o": mu_o,
    "rho_a": rho_a,
    "rho_o": rho_o,
    "D_a": D_a,
    "D_o": D_o,
    "d_si": dsi,
    "d_i": di,
    "d_o": do,
    "L": L,
    "epsilon": epsilon,
    "tau": tau,
    "l": l,
}

**Step 5.** Import experimental data and convert the flow rates to SI units.

In [6]:
datasets = {
    "exp_1": exp_data[exp_data["exp_id"]==1], # packing fraction
    "exp_2": exp_data[exp_data["exp_id"]==2], # flow rate
}

In [7]:
for name, df in datasets.items():
    df["Q_a_m3s"] = df["Q_a_mLmin"] * (1e-6 / 60)
    df["Q_o_m3s"] = df["Q_o_mLmin"] * (1e-6 / 60)

**Step 6.** Determine, from experiments, which runs are for regression, and which runs are for evaluation. For example, in the experiment labeled `fit_phi_only`, regression and evaluation are performed with only the packing fraction data, `exp_1`.

In [8]:
experiments = {  

    "fit_phi_only": {
        "regression": ["exp_1"],
        "evaluation": ["exp_1"]
    },

    "predict_flow": {
        "evaluation": ["exp_2"]
    }
}

## Parameterization

The model was first fit to the data from Experiment 1 `exp_1` under different parameter-fitting scenarios. The resulting fits were compared to determine how well each scenario reproduced the Experiment 1 `exp_2` data. The scenarios are shown in the table below.

| Case | Free parameter(s) | Interpretation | 
| :----- | :----- | :----- |
| I | $A$ | Fit $A$ while keeping $B$, $\alpha$, and $\beta$ fixed |
| II | $B$ | Fit $B$ while keeping $A$, $\alpha$, and $\beta$ fixed |
| III | $A$, $\alpha$ | Fit $A$ and alpha while keeping $B$ and $\beta$ fixed |
| IV | $A$, $B$ | Fit $A$ and $B$ while keeping $\alpha$ and $\beta$ fixed |
| V | $A$, $B$, $\alpha$ | Fit $A$, $B$, and $\alpha$ while keeping $\beta$ fixed |

The cases dictionary specifies which parameters are fixed and which are allowed to be fitted.

- `"B": 1.0` = $B$ is fixed at 1.0.
- `"A": None` = $A$ is not fixed, so `fit_model()` is allowed to estimate $A$ from the regression data.

For example, in Case I:
```
{
    "A": None,
    "B": 1.0,
    "alpha": 1.0,
    "beta": 0.60
}
```
only $A$ is fitted. $B$, $\alpha$, and $\beta$ are held constant.

In [9]:
cases = {

    "A_only": {
        "A": None,
        "B": 1.0,
        "alpha": 1.0,
        "beta": 0.60
    },

    "B_only": {
        "A": 1.0,
        "B": None,
        "alpha": 1.0,
        "beta": 0.60
    },

    "A_alpha": {
        "A": None,
        "B": 1.0,
        "alpha": None,
        "beta": 0.60
    },

    "A_B": {
        "A": None,
        "B": None,
        "alpha": 1.0,
        "beta": 0.60
    },

    "A_B_alpha": {
        "A": None,
        "B": None,
        "alpha": None,
        "beta": 0.60
    }
}

case_labels = {
    "A_only": ("I", "A"),
    "B_only": ("II", "B"),
    "A_alpha": ("III", "A, alpha"),
    "A_B": ("IV", "A, B"),
    "A_B_alpha": ("V", "A, B, alpha")
}

## Modeling $\phi$

For each case, the free parameters are estimated using the regression datasets via `fit_model()`. Then, the fitted parameters are evaluated on the separate evaluation datasets using `evaluate_model()`. The resulting table contains the parameter estimates and the evaluation $R²$ for every case.

The final `phi_results` table provides a side-by-side comparison of which parameters were allowed to vary, what values were fitted, and how well each resulting model performed on the evaluation data.

In [10]:
experiment = experiments["fit_phi_only"]

regression_sets, evaluation_sets = get_dataset_split(
    datasets,
    experiment
)

phi_results = []
fitted_params = {}

for name, fixed in cases.items():

    params, result = fit_model(
        regression_sets,
        fixed,
        constants
    )

    r2, _ = evaluate_model(
        params,
        evaluation_sets,
        constants
    )

    fitted_params[name] = params

    phi_results.append({
        "Case": case_labels[name][0],
        "Free parameter": case_labels[name][1],
        "A": params["A"],
        "B": params["B"],
        "alpha": params["alpha"],
        "beta": params["beta"],
        "R²": r2
    })

phi_results = pd.DataFrame(phi_results).sort_values("Case", ascending=True)
phi_results

,Case,Free parameter,A,B,alpha,beta,R²
0,I,A,1.006343,1.000000,1.000000,0.6,0.886590
1,II,B,1.000000,0.946105,1.000000,0.6,0.915618
2,III,"A, alpha",0.852336,1.000000,0.955117,0.6,0.969858
3,IV,"A, B",0.971392,0.780385,1.000000,0.6,0.962858
4,V,"A, B, alpha",0.270976,3.042130,0.556641,0.6,1.000000


From the above model analysis, the model with the highest goodness-of-fit score ($R²$) is Case V, which allowed $A$, $B$, and $\alpha$ to vary. This is consistent with the paper, which reports the best $R²$ value for Case V. We can calculate a delta value to show the difference in $R²$ values for the two implementations.

In [11]:
paper_r2_phi_only = [0.8745, 0.8791, 0.9593, 0.9407, 0.9938]

phi_results["ΔR²"] = phi_results["R²"] - paper_r2_phi_only
phi_results["ΔR²"] = phi_results["ΔR²"].map(lambda x: f"{x:+.4f}")

phi_results

,Case,Free parameter,A,B,alpha,beta,R²,ΔR²
0,I,A,1.006343,1.000000,1.000000,0.6,0.886590,+0.0121
1,II,B,1.000000,0.946105,1.000000,0.6,0.915618,+0.0365
2,III,"A, alpha",0.852336,1.000000,0.955117,0.6,0.969858,+0.0106
3,IV,"A, B",0.971392,0.780385,1.000000,0.6,0.962858,+0.0222
4,V,"A, B, alpha",0.270976,3.042130,0.556641,0.6,1.000000,+0.0062


The selected model was then applied to the data from Experiment 2 `exp_2`, in which the flow rate $v_{a}$ was adjusted. 

In [12]:
experiment = experiments["predict_flow"]

_, flow_evaluation = get_dataset_split(
    datasets,
    experiment
)

flow_results = []

for case_name in [
    "A_alpha",
    "A_B",
    "A_B_alpha"
]:

    params = fitted_params[case_name]

    r2, results = evaluate_model(
        params,
        flow_evaluation,
        constants
    )

    flow_results.append({
        "Case": case_labels[case_name][0],
        "Free parameter": case_labels[case_name][1],
        "R²": r2
    })

flow_results = pd.DataFrame(flow_results)
flow_results

,Case,Free parameter,R²
0,III,"A, alpha",0.754253
1,IV,"A, B",0.754302
2,V,"A, B, alpha",0.782689


The $R²$ values decreased across Cases III-V, reflecting the original published conclusion that the packing-fraction-only model fails to reflect the dynamic reduction in boundary layer thickness induced by higher velocities. As with the packing fraction dataset, we can calculate the delta value to show the difference in $R²$ values for the two implementations. These experiments are described in Section 4.2.2 of the publication.

In [13]:
paper_r2_flow = [0.7452, 0.7794, 0.7911]

flow_results["ΔR²"] = flow_results["R²"] - paper_r2_flow
flow_results["ΔR²"] = flow_results["ΔR²"].map(lambda x: f"{x:+.4f}")

flow_results

,Case,Free parameter,R²,ΔR²
0,III,"A, alpha",0.754253,+0.0091
1,IV,"A, B",0.754302,-0.0251
2,V,"A, B, alpha",0.782689,-0.0084


## Modeling $\phi$ and $v_{a}$

The reduction in $R²$ observed when applying the packing-fraction-based model to the experiment with velocity changes motivated the development of a new model that incorporates both effects in the regression. The model parameters were then re-optimized, using the best-fit parameters from the previous model (Case V) as the initial parameter values. Goodness of fit was subsequently re-evaluated under different parameter optimization regimes. Here, we present results from one case study, Case VIII, in which $A$ and $\beta$ were fitted simultaneously.

In [14]:
selected_model = fitted_params["A_B_alpha"]
selected_model

{'A': np.float64(0.27097618734771795),
 'B': np.float64(3.0421300732065535),
 'alpha': np.float64(0.5566411384134597),
 'beta': 0.6}

In [15]:
case8_params = modify_case(
    selected_model,
    A=None,
    beta=None
    )

In [16]:
regression_sets = [
    datasets["exp_1"],
    datasets["exp_2"]
]

In [17]:
parameter_results = []

x0 = [
    selected_model[param]
    for param in ["A", "B", "alpha", "beta"]
    if case8_params[param] is None
]

params, result = fit_model(
    regression_sets,
    case8_params,
    constants,
    x0=x0
)

case8_result = params.copy()

parameter_results.append({
    "Case": "VIII",
    "Free parameter": "A, beta",
    "A": params["A"],
    "B": params["B"],
    "alpha": params["alpha"],
    "beta": params["beta"]
})

parameter_results = pd.DataFrame(parameter_results)

In [18]:
r2_results = []

r2_phi, _ = evaluate_model(
    params,
    [datasets["exp_1"]],
    constants
)

r2_nu, _ = evaluate_model(
    params,
    [datasets["exp_2"]],
    constants
)

r2_total, _ = evaluate_model(
    params,
    [datasets["exp_1"], datasets["exp_2"]],
    constants
)

r2_results.append({
    "Case": "VIII",
    "R²_φ": r2_phi,
    "R²_flow": r2_nu,
    "R²_total": r2_total
})

r2_results = pd.DataFrame(r2_results)

merged_results = pd.merge(
    parameter_results,
    r2_results,
    on="Case"
)

merged_results

,Case,Free parameter,A,B,alpha,beta,R²_φ,R²_flow,R²_total
0,VIII,"A, beta",0.39389,3.04213,0.556641,0.482722,0.999475,0.99923,0.999238


Overall, increasing the flexibility of the model by allowing additional parameters to vary resulted in improved goodness of fit.